# Peptide retention time — saliency analysis

This notebook visualises the outputs of the PeptideGNN trained to predict
peptide retention time (RT). It reads saved results from output folder into a single results dataframe that contains
observed and predicted RT values together with per-atom saliency values, and
produces the four figures described below.

| Section | What it shows |
|---|---|
| 1. Model performance | Observed vs. predicted RT scatter plots with R, MAE, Δt₉₅% |
| 2. Saliency maps | Atomic saliency maps drawings plotted for multiple datasets (qualitative and quantitative) |
| 3. Amino-acid contributions | Mean residue-level saliency per dataset — which amino acids drive RT |
| 4. Neighbour effects | How a residue's RT contribution changes depending on its amino acid sequence neighbours |

**How to use:** edit only the cells marked `⚙ SETTINGS`. All other cells can be run as-is.

---
## Setup

Imports and helper functions.

In [ ]:
import os

os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

import logging
import molgraph
import tensorflow as tf

from vis_util import (
    DEFAULT_OUTPUT_DIR,
    DATASET_ORDER,
    read_result,
    prepare_datasets,
    add_aa_saliency,
    normalize_saliency_dataset_wise,
    get_common_peptides,
    plot_performance,
    plot_peptide_maps,
    plot_aa_bar_charts,
    plot_neighbor_heatmaps,
    plot_comparison_chart,
)

logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_visible_devices(gpus[0], 'GPU')
    tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=8000)])

---
## ⚙ SETTINGS — edit this cell before running anything else

In [ ]:
# ── Path to the folder that contains *_prediction.csv and *_saliency.pkl files ──
OUTPUT_DIR = DEFAULT_OUTPUT_DIR

# ── Folder where saliency map images will be saved ──
SALIENCY_MAP_DIR = DEFAULT_OUTPUT_DIR / 'saliency-maps/'

# ── Dataset names to include (set to None to use all datasets found) ──
SELECTED_DATASETS = ["SWATH Library", "LUNA HILIC", "SCX"]      # e.g. ["SWATH Library", "LUNA HILIC", "SCX"]

# ── Which data split to analyse ──
#    "test"  → held-out test set only (recommended for performance plots)
#    "train" → training set
#    None    → all rows (recommended for all other plots)
SET_TYPE = "test"

# ── Comparison chart: choose two datasets to compare side by side ──
COMPARISON_LEFT  = "SWATH Library"
COMPARISON_RIGHT = "LUNA HILIC"

# ── Neighbour effect analysis: sequence distances to inspect ──
NEIGHBOUR_POSITIONS  = [1, 5, 10]       # distance from studied residue
NEIGHBOUR_DIRECTIONS = ["N-term", "C-term"]

---
## Load data

In [ ]:
# Read all prediction + saliency files into one flat DataFrame.
# Columns: dataset | peptide | observed_rt | predicted_rt | set_type
#          saliency | dataset_normalized_saliency
result = read_result(OUTPUT_DIR)
result = prepare_datasets(result, selected_datasets=None, order=DATASET_ORDER)
result = normalize_saliency_dataset_wise(result)

# Segment-sum atomic saliency → residue-level saliency (~6 min).
# Adds an 'amino_acid_saliency' column to result. Run once per session.
# Optionally, to avoid recomputing, save and reload:
#   result.to_pickle('result_with_aa.pkl')
#   result = pd.read_pickle('result_with_aa.pkl')
result = add_aa_saliency(result)

print(f"Loaded {len(result):,} rows across {result['dataset'].nunique()} datasets.")
result.head()

---
## 1. Model performance

Scatter plots of observed vs. predicted RT for every dataset on test set.
Each panel shows Pearson R, mean absolute error (MAE), and the 95 % prediction
interval width (Δt₉₅%). Dashed red lines mark the 99th-percentile error.

In [ ]:
plot_performance(
    result[result["set_type"] == SET_TYPE] if SET_TYPE else result,
    order=DATASET_ORDER,
)

---
## 2. Saliency maps

Two rendering modes per peptide:

- **Qualitative** — each molecule is independently colour-scaled, making
  easy to visualize within-peptide interactions sides in LC.
- **Quantitative** — all molecules share a global colour scale, allowing
  direct cross-dataset comparison of saliency magnitudes.

Images are saved to `SALIENCY_MAP_DIR`.

In [ ]:
# Find peptides present in every selected dataset.
# Passing SELECTED_DATASETS ensures we only intersect the datasets you care about,
# not all datasets in the file. If SELECTED_DATASETS is None, all datasets are used.
# To plot specific peptides instead, replace this with a manual list, e.g.:
#   common_peptides = ["PEPTIDEK", "ACDEFGHIK"]
common_peptides = sorted(get_common_peptides(result, datasets=SELECTED_DATASETS))
print(f"{len(common_peptides)} peptides found in all selected datasets.")

In [ ]:
plot_peptide_maps(
    df_qual=result if SELECTED_DATASETS is None else result[result["dataset"].isin(SELECTED_DATASETS)],          # uses raw saliency column
    df_quant=result if SELECTED_DATASETS is None else result[result["dataset"].isin(SELECTED_DATASETS)],          # uses dataset_normalized_saliency column
    peptides=common_peptides,
    output_folder=SALIENCY_MAP_DIR,
)

---
## 3. Amino-acid contributions

For each dataset, bars show the mean residue-level saliency per amino acid
type (sorted by polarity, nonpolar → polar).  
🟢 Green = positive saliency (residue increases predicted RT)  
🔴 Red = negative saliency (residue decreases predicted RT)

In [ ]:
plot_aa_bar_charts(
    result,
    aa_saliency_col='amino_acid_saliency',
    dataset_order=DATASET_ORDER,           
    n_cols=5,                            
    set_type=None,
)

---
## 4. Neighbour effects

Heatmaps showing how the saliency of residue **aa1** (rows) is influenced
by the identity of its neighbour **aa2** (columns) at positions
±1, ±5, and ±10 in the sequence.

One figure is produced per dataset; panels within each figure share the
same colour scale so near- and long-range effects can be compared.

In [ ]:
plot_neighbor_heatmaps(
    result,
    aa_saliency_col='amino_acid_saliency',
    positions=NEIGHBOUR_POSITIONS,
    directions=NEIGHBOUR_DIRECTIONS,
    set_type=None,
)

---
## 5. Comparison chart

A paired heatmap placing two datasets side by side on the same peptides
and colour scale.  Useful for spotting which residues drive differences
in RT between chromatographic conditions.

In [ ]:
plot_comparison_chart(
    result,
    left=COMPARISON_LEFT,
    right=COMPARISON_RIGHT,
    max_peptides=20,
    set_type=None,
)